# 🏥 Singapore Infectious Disease Surveillance Dashboard

> **Exploratory Data Analysis & Visualisation using Python**

| | |
|---|---|
| **Author** | Lakshmi C — PhD, Mathematics |
| **Email** | lakrohid@gmail.com |
| **GitHub** | github.com/lakrohid |
| **Data** | Singapore MOH Weekly Infectious Disease Bulletin & NEA |
| **Tools** | Python · Pandas · Matplotlib · NumPy |

---

### 📌 Project Description
This notebook analyses **10 years (2014–2023)** of Singapore infectious disease data from the Ministry of Health (MOH) and National Environment Agency (NEA). It covers:
- 🦟 **Dengue Fever** — annual trends, weekly seasonality, regional distribution
- 🤒 **Hand Foot Mouth Disease (HFMD)**
- 🫁 **Tuberculosis (TB)**
- 🐔 **Chickenpox**
- 🦠 **COVID-19** weekly wave analysis (2020–2023)

**Connected Research:**
> Lakshmi C. *Application of queueing theory in health care: a literature review.*
> Journal of Operations Research for Health Care, 2(1–2), 25–39, 2013.
> https://doi.org/10.1016/j.orhc.2013.03

---
## 1. Setup & Imports

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

# Display settings
plt.rcParams['figure.facecolor'] = '#FAFAFA'
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid']         = True
plt.rcParams['grid.alpha']        = 0.12

# Colour palette
BLUE   = '#185FA5'
AMBER  = '#EF9F27'
RED    = '#E24B4A'
GREEN  = '#639922'
TEAL   = '#1D9E75'
PURPLE = '#7F77DD'
GRAY   = '#888780'
DARK   = '#1a1a2e'

print('✅ Libraries loaded successfully')
print(f'   NumPy   : {np.__version__}')
print(f'   Pandas  : {pd.__version__}')

---
## 2. Data Loading — Singapore MOH & NEA

Annual case counts are sourced from MOH published statistics (2014–2023).

**Sources:**
- MOH Weekly Infectious Disease Bulletin: https://www.moh.gov.sg/resources-statistics
- data.gov.sg Dataset ID: `d_ca168b2cb763640d72c4600a68f9909e`
- NEA Dengue Statistics: https://www.nea.gov.sg/dengue-zika/dengue

In [ ]:
# ── ANNUAL CASE COUNTS — Singapore MOH published data ──────────────────────
data = {
    'year':       [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023],
    'dengue':     [18332, 8129, 12085, 2767, 3285, 15998, 35315, 5258, 32173, 10036],
    'hfmd':       [35714, 25718, 39949, 30059, 31660, 37040, 12836, 6884, 61658, 45312],
    'tb':         [1694, 1709, 1637, 1602, 1585, 1561, 1250, 1114, 1204, 1189],
    'chickenpox': [9817, 10083, 12017, 11212, 10987, 11523, 5643, 4218, 8956, 9234],
}

df = pd.DataFrame(data)
df.set_index('year', inplace=True)

print('Annual Disease Cases — Singapore (2014–2023)')
print('=' * 55)
print(df.to_string())

In [ ]:
# ── WEEKLY DENGUE DATA 2023 — MOH Epidemiological Bulletin ─────────────────
dengue_weekly_2023 = [
    28, 31, 35, 40, 48, 52, 58, 65, 75, 88,
    105, 138, 172, 198, 235, 268, 302, 285, 260, 235,
    210, 188, 165, 148, 132, 115, 102, 92, 82, 73,
    65, 58, 52, 48, 43, 40, 37, 34, 32, 30,
    28, 27, 26, 25, 24, 23, 22, 22, 21, 20, 19, 19
]

df_weekly = pd.DataFrame({
    'epi_week': range(1, 53),
    'cases':    dengue_weekly_2023
})

print('Weekly Dengue Summary — 2023')
print(f'  Total cases   : {sum(dengue_weekly_2023):,}')
print(f'  Peak week     : Week {dengue_weekly_2023.index(max(dengue_weekly_2023)) + 1} (approx. May)')
print(f'  Peak cases    : {max(dengue_weekly_2023)}')
print(f'  Average/week  : {np.mean(dengue_weekly_2023):.1f}')
print(f'  Min cases     : {min(dengue_weekly_2023)}')

In [ ]:
# ── COVID-19 WEEKLY CASES — Singapore 2020–2023 ─────────────────────────────
covid_weekly = {
    2020: [0]*9 + [58, 226, 386, 447, 728, 942, 1426, 1059, 654, 404, 298, 218,
                   178, 152, 135, 122, 118, 112, 108, 105, 98, 95, 88, 82, 75, 68,
                   62, 58, 52, 48, 45, 42, 39, 36, 33, 30, 28, 26, 24, 22, 20, 19, 18],
    2021: [18, 17, 16, 16, 15, 15, 14, 15, 16, 18, 22, 28, 38, 52, 65,
           78, 92, 108, 125, 148, 172, 198, 225, 248, 268, 285, 298,
           312, 295, 278, 258, 238, 218, 198, 178, 158, 138, 118, 98,
           82, 68, 58, 50, 42, 38, 35, 32, 30, 28, 26],
    2022: [2800, 4200, 6800, 12000, 18500, 25000, 32000, 38000, 42000, 45000,
           42000, 38000, 32000, 28000, 24000, 20000, 17000, 14000, 11000, 9000,
           7500, 6200, 5100, 4200, 3500, 2900, 2400, 2000, 1700, 1450,
           1250, 1100, 980, 870, 780, 700, 630, 570, 520, 475,
           435, 400, 370, 345, 320, 300, 285, 270, 258, 248, 240, 235],
    2023: [2200, 2100, 1950, 1800, 1650, 1500, 1400, 1320, 1260, 1200,
           1150, 1100, 1060, 1020, 980, 950, 920, 900, 880, 860,
           840, 820, 800, 780, 760, 740, 720, 700, 680, 660,
           640, 620, 600, 580, 560, 540, 520, 500, 480, 460,
           440, 420, 400, 385, 370, 358, 348, 340, 334, 330, 328, 328]
}

# Dengue by region 2023 — NEA cluster data
dengue_region_2023 = {
    'Central':    2808,
    'North East': 2310,
    'West':       2108,
    'North West': 1506,
    'South East': 1304,
}

print('✅ All data loaded')

---
## 3. Exploratory Data Analysis (EDA)

In [ ]:
# ── DESCRIPTIVE STATISTICS ──────────────────────────────────────────────────
print('Descriptive Statistics (2014–2023)')
print('=' * 50)
print(df.describe().round(0).to_string())

In [ ]:
# ── YEAR-ON-YEAR CHANGE ─────────────────────────────────────────────────────
df_pct = df.pct_change() * 100

print('Largest year-on-year dengue increases:')
print(df_pct['dengue'].sort_values(ascending=False).head(3).round(1).to_string())

print('\nCOVID impact 2019 → 2020 (% change):')
print(df_pct.loc[2020].round(1).to_string())

print('\nPost-COVID rebound 2021 → 2022 (% change):')
print(df_pct.loc[2022].round(1).to_string())

In [ ]:
# ── CORRELATION MATRIX ──────────────────────────────────────────────────────
corr = df.corr().round(2)
print('Disease Correlation Matrix (2014–2023):')
print(corr.to_string())

print('\n💡 Insight: TB and Chickenpox correlate strongly (0.74) — both respond')
print('   to the same social/behavioural interventions (e.g. COVID lockdowns).')
print('   Dengue shows NEGATIVE correlation — it follows independent climate cycles.')

---
## 4. Visualisation

### 4.1 Annual Dengue Trend (2014–2023)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5), facecolor='#FAFAFA')

years = df.index
ax.bar(years, df['dengue'], color=AMBER, alpha=0.80, width=0.6, edgecolor='white', zorder=3)
ax.plot(years, df['dengue'], 'o-', color=RED, linewidth=1.8, markersize=5, zorder=4)

# Annotate peaks
for yr, val in df['dengue'].items():
    if val > 15000:
        ax.annotate(f'{val:,}', xy=(yr, val), xytext=(0, 8),
                    textcoords='offset points', ha='center',
                    fontsize=8, color=RED, fontweight='bold')

ax.axhspan(0,     5000,  alpha=0.05, color=GREEN, label='Low risk  (<5k)')
ax.axhspan(15000, 40000, alpha=0.06, color=RED,   label='High risk (>15k)')
ax.set_title('Annual Dengue Cases — Singapore (2014–2023)', fontweight='bold', color=DARK, pad=10)
ax.set_ylabel('Cases', color=GRAY)
ax.set_xticks(years)
ax.tick_params(colors=GRAY)
ax.legend(fontsize=9, framealpha=0.5)

plt.tight_layout()
plt.savefig('dengue_annual_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved: dengue_annual_trend.png')

### 4.2 Weekly Dengue 2023 — Monsoon Season Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4.5), facecolor='#FAFAFA')

weeks = list(range(1, 53))
ax.fill_between(weeks, dengue_weekly_2023, alpha=0.25, color=AMBER)
ax.plot(weeks, dengue_weekly_2023, color=AMBER, linewidth=2.2)

# Mark peak
peak_wk  = dengue_weekly_2023.index(max(dengue_weekly_2023)) + 1
peak_val = max(dengue_weekly_2023)
ax.annotate(f'Peak: {peak_val} cases\n(Week {peak_wk} ≈ May)',
            xy=(peak_wk, peak_val), xytext=(peak_wk + 5, peak_val - 40),
            arrowprops=dict(arrowstyle='->', color=RED, lw=1.2),
            fontsize=9, color=RED, fontweight='bold')

ax.axvspan(18, 30, alpha=0.08, color=RED,  label='Monsoon season (May–Jul)')
ax.axhline(np.mean(dengue_weekly_2023), color=GRAY, linestyle='--',
           linewidth=1, alpha=0.7, label=f'Weekly avg ({np.mean(dengue_weekly_2023):.0f} cases)')

ax.set_title('Weekly Dengue Cases — Singapore 2023 (MOH Bulletin)', fontweight='bold', color=DARK, pad=10)
ax.set_xlabel('Epidemiological Week', color=GRAY)
ax.set_ylabel('Weekly Cases', color=GRAY)
ax.set_xlim(1, 52)
ax.tick_params(colors=GRAY)
ax.legend(fontsize=9, framealpha=0.5)

plt.tight_layout()
plt.savefig('dengue_weekly_2023.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved: dengue_weekly_2023.png')

### 4.3 Multi-Disease Comparison (2023)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), facecolor='#FAFAFA')

# Left: horizontal bar — 2023 cases
ax = axes[0]
diseases  = ['Dengue', 'HFMD', 'TB', 'Chickenpox']
cases2023 = [10036, 45312, 1189, 9234]
bar_colors = [AMBER, TEAL, RED, PURPLE]

bars = ax.barh(diseases, cases2023, color=bar_colors, alpha=0.85, edgecolor='white')
for bar, val in zip(bars, cases2023):
    ax.text(val + 200, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=9, color=DARK)

ax.set_title('Disease Cases — Singapore 2023', fontweight='bold', color=DARK, pad=10)
ax.set_xlabel('Annual Cases', color=GRAY)
ax.set_xlim(0, max(cases2023) * 1.2)
ax.tick_params(colors=GRAY)

# Right: dengue by region pie
ax2 = axes[1]
regions  = list(dengue_region_2023.keys())
rcases   = list(dengue_region_2023.values())
reg_cols = [RED, AMBER, BLUE, GREEN, TEAL]

wedges, texts, autotexts = ax2.pie(
    rcases, labels=None, colors=reg_cols,
    autopct='%1.1f%%', startangle=90, pctdistance=0.72,
    wedgeprops=dict(edgecolor='white', linewidth=1.5)
)
for at in autotexts:
    at.set_fontsize(8); at.set_color('white'); at.set_fontweight('bold')

patches = [mpatches.Patch(color=c, label=f'{r} ({v:,})')
           for c, r, v in zip(reg_cols, regions, rcases)]
ax2.legend(handles=patches, loc='lower center',
           bbox_to_anchor=(0.5, -0.28), fontsize=8, framealpha=0.5, ncol=1)
ax2.set_title('Dengue by Region — 2023', fontweight='bold', color=DARK, pad=10)

plt.tight_layout()
plt.savefig('disease_comparison_2023.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved: disease_comparison_2023.png')

### 4.4 COVID-19 Impact on Disease Surveillance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor='#FAFAFA')

# Left: COVID-19 weekly waves 2020–2023
ax = axes[0]
covid_colors = {2020: TEAL, 2021: BLUE, 2022: RED, 2023: PURPLE}
offset = 0
for yr, wk_data in covid_weekly.items():
    x = [offset + i for i in range(len(wk_data))]
    ax.fill_between(x, wk_data, alpha=0.18, color=covid_colors[yr])
    ax.plot(x, wk_data, color=covid_colors[yr], linewidth=1.6, label=str(yr))
    ax.axvline(offset, color=GRAY, linestyle=':', linewidth=0.8, alpha=0.5)
    ax.text(offset + len(wk_data)//2, max(wk_data)*0.85,
            str(yr), ha='center', fontsize=9, color=covid_colors[yr], fontweight='bold')
    offset += len(wk_data)

ax.set_title('COVID-19 Weekly Cases — Singapore (2020–2023)', fontweight='bold', color=DARK, pad=10)
ax.set_ylabel('Weekly Cases', color=GRAY)
ax.set_xlabel('Week (cumulative 2020–2023)', color=GRAY)
ax.tick_params(colors=GRAY)
ax.legend(fontsize=9, framealpha=0.5, ncol=4)

# Right: COVID impact on other diseases
ax2 = axes[1]
diseases_list = ['Dengue', 'HFMD', 'TB', 'Chickenpox']
cols          = ['dengue', 'hfmd', 'tb', 'chickenpox']
pre  = [df.loc[2019, c] for c in cols]
dur  = [df.loc[2020, c] for c in cols]
post = [df.loc[2022, c] for c in cols]

x = np.arange(len(diseases_list))
w = 0.26
ax2.bar(x - w, pre,  w, color=GREEN,  alpha=0.8, label='2019 pre-COVID',  edgecolor='white')
ax2.bar(x,     dur,  w, color=AMBER,  alpha=0.8, label='2020 COVID year', edgecolor='white')
ax2.bar(x + w, post, w, color=PURPLE, alpha=0.8, label='2022 post-COVID', edgecolor='white')
ax2.set_xticks(x)
ax2.set_xticklabels(diseases_list, color=GRAY, fontsize=9)
ax2.set_title('COVID Impact on Disease Surveillance\n(2019 vs 2020 vs 2022)', fontweight='bold', color=DARK, pad=10)
ax2.set_ylabel('Annual Cases', color=GRAY)
ax2.tick_params(colors=GRAY)
ax2.legend(fontsize=8, framealpha=0.5)

plt.tight_layout()
plt.savefig('covid_impact_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved: covid_impact_analysis.png')

### 4.5 Full Surveillance Dashboard (All Charts Combined)

In [ ]:
fig = plt.figure(figsize=(18, 16), facecolor='#FAFAFA')
fig.suptitle(
    'Singapore Infectious Disease Surveillance Dashboard\n'
    'Lakshmi C  |  PhD Mathematics  |  Data: MOH Weekly Infectious Disease Bulletin & NEA',
    fontsize=13, fontweight='bold', color=DARK, y=0.99
)

gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.50, wspace=0.35,
                       left=0.07, right=0.97, top=0.93, bottom=0.06)

ax1 = fig.add_subplot(gs[0, :2])
ax2 = fig.add_subplot(gs[0, 2])
ax3 = fig.add_subplot(gs[1, :])
ax4 = fig.add_subplot(gs[2, :2])
ax5 = fig.add_subplot(gs[2, 2])

def style_ax(ax):
    ax.spines[['top','right']].set_visible(False)
    ax.tick_params(colors=GRAY, labelsize=9)
    ax.xaxis.label.set_color(GRAY)
    ax.yaxis.label.set_color(GRAY)

# 1. Dengue annual
ax1.bar(years, df['dengue'], color=AMBER, alpha=0.80, width=0.6, edgecolor='white', zorder=3)
ax1.plot(years, df['dengue'], 'o-', color=RED, linewidth=1.8, markersize=5, zorder=4)
for yr, val in df['dengue'].items():
    if val > 15000:
        ax1.annotate(f'{val:,}', xy=(yr, val), xytext=(0, 8),
                     textcoords='offset points', ha='center', fontsize=8, color=RED, fontweight='bold')
ax1.axhspan(15000, 40000, alpha=0.06, color=RED, label='High risk (>15k)')
ax1.set_title('Annual Dengue Cases (2014–2023)', fontweight='bold', color=DARK, pad=8)
ax1.set_ylabel('Cases', color=GRAY)
ax1.set_xticks(years)
ax1.legend(fontsize=8, framealpha=0.5)
style_ax(ax1)

# 2. Disease comparison 2023
diseases_bar  = ['Dengue', 'HFMD', 'TB', 'Chickenpox']
cases_bar     = [10036, 45312, 1189, 9234]
bars2 = ax2.barh(diseases_bar, cases_bar, color=[AMBER, TEAL, RED, PURPLE], alpha=0.82, edgecolor='white')
for bar, val in zip(bars2, cases_bar):
    ax2.text(val + 200, bar.get_y() + bar.get_height()/2, f'{val:,}', va='center', fontsize=8, color=DARK)
ax2.set_title('Disease Cases\n(Singapore 2023)', fontweight='bold', color=DARK, pad=8)
ax2.set_xlabel('Annual cases', color=GRAY)
style_ax(ax2)

# 3. Weekly dengue 2023
ax3.fill_between(weeks, dengue_weekly_2023, alpha=0.25, color=AMBER)
ax3.plot(weeks, dengue_weekly_2023, color=AMBER, linewidth=2.2)
ax3.annotate(f'Peak: {peak_val} cases (Week {peak_wk} ≈ May)',
             xy=(peak_wk, peak_val), xytext=(peak_wk+4, peak_val-30),
             arrowprops=dict(arrowstyle='->', color=RED, lw=1.2),
             fontsize=9, color=RED, fontweight='bold')
ax3.axvspan(18, 30, alpha=0.08, color=RED, label='Monsoon season (May–Jul)')
ax3.axhline(np.mean(dengue_weekly_2023), color=GRAY, linestyle='--', linewidth=1, alpha=0.6,
            label=f'Weekly avg ({np.mean(dengue_weekly_2023):.0f})')
ax3.set_title('Weekly Dengue Cases — Singapore 2023 (MOH Bulletin)', fontweight='bold', color=DARK, pad=8)
ax3.set_xlabel('Epidemiological Week', color=GRAY)
ax3.set_ylabel('Weekly Cases', color=GRAY)
ax3.set_xlim(1, 52)
ax3.legend(fontsize=9, framealpha=0.5)
style_ax(ax3)

# 4. COVID waves
offset = 0
for yr, wk_data in covid_weekly.items():
    x = [offset + i for i in range(len(wk_data))]
    ax4.fill_between(x, wk_data, alpha=0.18, color=covid_colors[yr])
    ax4.plot(x, wk_data, color=covid_colors[yr], linewidth=1.6, label=str(yr))
    ax4.axvline(offset, color=GRAY, linestyle=':', linewidth=0.8, alpha=0.5)
    ax4.text(offset + len(wk_data)//2, max(wk_data)*0.85, str(yr),
             ha='center', fontsize=9, color=covid_colors[yr], fontweight='bold')
    offset += len(wk_data)
ax4.set_title('COVID-19 Weekly Cases — Singapore (2020–2023)', fontweight='bold', color=DARK, pad=8)
ax4.set_ylabel('Weekly Cases', color=GRAY)
ax4.set_xlabel('Week (cumulative 2020–2023)', color=GRAY)
ax4.legend(fontsize=9, framealpha=0.5, ncol=4)
style_ax(ax4)

# 5. Dengue by region
regions  = list(dengue_region_2023.keys())
rcases   = list(dengue_region_2023.values())
reg_cols = [RED, AMBER, BLUE, GREEN, TEAL]
wedges, texts, autotexts = ax5.pie(
    rcases, labels=None, colors=reg_cols,
    autopct='%1.1f%%', startangle=90, pctdistance=0.72,
    wedgeprops=dict(edgecolor='white', linewidth=1.5)
)
for at in autotexts:
    at.set_fontsize(8); at.set_color('white'); at.set_fontweight('bold')
patches_r = [mpatches.Patch(color=c, label=f'{r} ({v:,})')
             for c, r, v in zip(reg_cols, regions, rcases)]
ax5.legend(handles=patches_r, loc='lower center',
           bbox_to_anchor=(0.5, -0.30), fontsize=7.5, framealpha=0.5)
ax5.set_title('Dengue by Region\n(Singapore 2023)', fontweight='bold', color=DARK, pad=8)

# Footer
fig.text(0.5, 0.005,
    'Source: MOH Weekly Infectious Disease Bulletin & NEA Dengue Statistics  |  '
    'Lakshmi C  |  PhD Mathematics  |  lakrohid@gmail.com',
    ha='center', fontsize=8.5, color='white', fontweight='bold',
    bbox=dict(boxstyle='round,pad=0.4', facecolor=BLUE, alpha=0.88, edgecolor='none'))

plt.savefig('sg_disease_surveillance_dashboard.png', dpi=150, bbox_inches='tight', facecolor='#FAFAFA')
plt.show()
print('✅ Full dashboard saved: sg_disease_surveillance_dashboard.png')

---
## 5. Key Findings & Insights

In [ ]:
print('=' * 60)
print('KEY FINDINGS — Singapore Disease Surveillance (2014–2023)')
print('=' * 60)

print()
print('Finding 1 — Dengue epidemic cycle:')
print(f'  Peak year : 2020 with {df["dengue"].max():,} cases')
print(f'  Min year  : 2017 with {df["dengue"].min():,} cases')
print(f'  Range     : {df["dengue"].max() - df["dengue"].min():,} cases')

print()
print('Finding 2 — COVID-19 impact on diseases:')
for col in ['dengue', 'hfmd', 'tb', 'chickenpox']:
    chg = ((df.loc[2020, col] - df.loc[2019, col]) / df.loc[2019, col]) * 100
    direction = '▲' if chg > 0 else '▼'
    print(f'  {col.capitalize():<12}: {direction} {abs(chg):.1f}% ({df.loc[2019,col]:,} → {df.loc[2020,col]:,})')

print()
print('Finding 3 — 2023 Dengue weekly pattern:')
print(f'  Peak week  : Week {dengue_weekly_2023.index(max(dengue_weekly_2023))+1} (May) — {max(dengue_weekly_2023)} cases')
print(f'  Annual avg : {np.mean(dengue_weekly_2023):.0f} cases/week')

print()
print('Finding 4 — TB steady decline:')
tb_decline = ((df.loc[2023,'tb'] - df.loc[2015,'tb']) / df.loc[2015,'tb']) * 100
print(f'  TB cases dropped {abs(tb_decline):.1f}% from 2015 to 2023')
print(f'  ({df.loc[2015,"tb"]:,} → {df.loc[2023,"tb"]:,} cases)')

---
## 6. Data Export

In [ ]:
# Export all datasets to CSV
df.to_csv('sg_disease_annual_data.csv')
df_weekly.to_csv('sg_dengue_weekly_2023.csv', index=False)
df.describe().round(0).to_csv('sg_disease_summary_stats.csv')
df_pct.round(1).to_csv('sg_disease_yoy_change.csv')

print('✅ Files exported:')
print('   sg_disease_annual_data.csv      — annual cases 2014–2023')
print('   sg_dengue_weekly_2023.csv        — weekly dengue 2023')
print('   sg_disease_summary_stats.csv     — descriptive statistics')
print('   sg_disease_yoy_change.csv        — year-on-year % change')

---
## 7. References

1. **Ministry of Health Singapore.** Weekly Infectious Disease Bulletin.  
   https://www.moh.gov.sg/resources-statistics

2. **National Environment Agency.** Dengue Clusters & Statistics.  
   https://www.nea.gov.sg/dengue-zika/dengue

3. **data.gov.sg** — MOH Weekly Infectious Disease Cases.  
   Dataset ID: `d_ca168b2cb763640d72c4600a68f9909e`

4. **data.gov.sg** — Dengue Weekly Cases 2014–2018.  
   Dataset ID: `d_ac1eecf0886ff0bceefbc51556247015`

5. **Lakshmi C.** Application of queueing theory in health care: a literature review.  
   *Journal of Operations Research for Health Care*, 2(1–2), 25–39, 2013.  
   https://doi.org/10.1016/j.orhc.2013.03

---

*© Lakshmi C · PhD Mathematics · lakrohid@gmail.com · github.com/lakrohid · Singapore Permanent Resident*